<p align="left">
  <img src="assets/rwth_logo.png" width="80" style="vertical-align: middle; margin-right: 15px;">
  <span style="font-size: 32px; font-weight: bold; vertical-align: middle;">
      Machine Learning for Combinatorial Optimization
  </span>
</p>

## Starting kit

This notebook shows how to load the training data, inspect one graph, train a small baseline model, and produce predictions in the format expected by the competition.

Each graph is a weighted graph. For every node, the model predicts whether that node belongs to a solution for three tasks:

- `mis`: maximum independent set
- `mvc`: minimum vertex cover
- `mc`: maximum clique

## Load the training data

The file `train_data.pt` is stored as a PyTorch Geometric collated dataset. This means it is not a list of graphs. Instead, it contains one large `Data` object and a `slices` dictionary that stores the graph boundaries.

In [12]:
from pathlib import Path

import torch
from torch.utils.data import Dataset
from torch_geometric.data.separate import separate
from torch_geometric.loader import DataLoader

In [13]:
TRAIN_FILE = Path("train_data.pt")

data, slices = torch.load(TRAIN_FILE, weights_only=False)
num_graphs = int(slices["x"].numel() - 1)

print(f"Number of graphs: {num_graphs}")
print(data)

Number of graphs: 5000
Data(edge_index=[2, 5506190], num_nodes=372500, x=[372500], mis_obj=[5000], mis_sol=[372500], mvc_obj=[5000], mvc_sol=[372500], cli_obj=[5000], cli_sol=[372500])


The collated object concatenates all graph attributes. For example, `data.x` contains all node weights, while `slices["x"]` tells us where the node weights of each graph start and stop.

In [14]:
print("x slices:", slices["x"][:5])
print("edge_index slices:", slices["edge_index"][:5])

x slices: tensor([  0,  50, 100, 150, 200])
edge_index slices: tensor([   0,  684, 1524, 2710, 3032])


## Recover one graph

We can recover an individual graph using PyG's `separate` helper. In this dataset, the edge indices are already local to each graph, so we keep `decrement=False`.

In [15]:
def get_graph(idx):
    return separate(
        cls=data.__class__,
        batch=data,
        idx=idx,
        slice_dict=slices,
        decrement=False,
    )


graph = get_graph(0)
print(graph)
print("node weights:", graph.x.shape)
print("edges:", graph.edge_index.shape)
print("MIS labels:", graph.mis_sol.shape)
print("MVC labels:", graph.mvc_sol.shape)
print("MC labels:", graph.cli_sol.shape)

Data(edge_index=[2, 684], x=[50], mis_obj=[1], mis_sol=[50], mvc_obj=[1], mvc_sol=[50], cli_obj=[1], cli_sol=[50], num_nodes=50)
node weights: torch.Size([50])
edges: torch.Size([2, 684])
MIS labels: torch.Size([50])
MVC labels: torch.Size([50])
MC labels: torch.Size([50])


The important graph fields are:

- `x`: node weights
- `edge_index`: graph edges in PyG format
- `mis_sol`, `mvc_sol`, `cli_sol`: binary node labels for the three tasks
- `mis_obj`, `mvc_obj`, `cli_obj`: objective values of the reference solutions

## A small dataset wrapper

The wrapper below makes the collated file behave like a standard dataset.

In [16]:
class GraphDataset(Dataset):
    def __init__(self, path):
        self.data, self.slices = torch.load(path, weights_only=False)
        self.num_graphs = int(self.slices["x"].numel() - 1)

    def __len__(self):
        return self.num_graphs

    def __getitem__(self, idx):
        return separate(
            cls=self.data.__class__,
            batch=self.data,
            idx=idx,
            slice_dict=self.slices,
            decrement=False,
        )


dataset = GraphDataset(TRAIN_FILE)
print(len(dataset))
print(dataset[0])

5000
Data(edge_index=[2, 684], x=[50], mis_obj=[1], mis_sol=[50], mvc_obj=[1], mvc_sol=[50], cli_obj=[1], cli_sol=[50], num_nodes=50)


## Train a baseline GNN

The provided `model.py` contains a small GIN network. The training loop below is intentionally minimal. It trains node-level binary classifiers for the three tasks at once.

In [17]:
import torch.nn.functional as F

from model import GIN


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GIN(in_channels=1, hidden_channels=64, num_layers=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# A small subset keeps this demo quick. Use more graphs for a real run.
train_graphs = [dataset[i] for i in range(min(256, len(dataset)))]
loader = DataLoader(train_graphs, batch_size=16, shuffle=True)

for epoch in range(3):
    model.train()
    total_loss = 0.0

    for batch in loader:
        batch = batch.to(device)
        x = batch.x.float().view(-1, 1)
        y = torch.stack(
            [batch.mis_sol, batch.mvc_sol, batch.cli_sol],
            dim=-1,
        ).float()

        optimizer.zero_grad()
        logits = model(x, batch.edge_index)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs

    print(f"epoch {epoch + 1}: loss = {total_loss / len(train_graphs):.4f}")

epoch 1: loss = 0.6246
epoch 2: loss = 0.5163
epoch 3: loss = 0.4677


## Prediction format

At evaluation time, your submission receives one unlabeled graph at a time. The `predict` method must return a dictionary with one binary tensor per task.

In [18]:
@torch.no_grad()
def predict_one(model, graph):
    model.eval()
    graph = graph.to(device)
    x = graph.x.float().view(-1, 1)
    logits = model(x, graph.edge_index)

    return {
        "mis": (logits[:, 0] > 0).long().cpu(),
        "mvc": (logits[:, 1] > 0).long().cpu(),
        "mc": (logits[:, 2] > 0).long().cpu(),
    }


prediction = predict_one(model, dataset[0])
{key: value.shape for key, value in prediction.items()}

{'mis': torch.Size([50]), 'mvc': torch.Size([50]), 'mc': torch.Size([50])}

## Creating a submission

Your submission should contain a `model.py` file with a `Model` class. The evaluator will instantiate it and call `predict(data)` on each test graph.

The starting kit already provides this interface. You can replace the architecture, add preprocessing, or load trained weights from `model.pt`, but keep the return format of `predict` unchanged.

In [19]:
torch.save(model.state_dict(), "./model.pt")

For a stronger solution, consider adding better node features, training for longer, tuning the GNN architecture, and repairing infeasible predictions before returning them.